# 01 Data Cleaning
This is where we work through cleaning up our data, normalizing, etc.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# GOAL: We want to take the dataset, and export a cleaned version of the csv to our processed data folder.

# Find the working directory for path 
Path.cwd() # We dont need this in the .py file probably, this was just for this starting bit

# First we load the dataset into the notebook in a pandas dataframe, and view the first 20 rows.
df_original = pd.read_csv("../data/raw/Meteorite_Landings.csv")
#df_original.head(20) # uncomment if need be

df = df_original.copy() # We make a copy of the original dataframe, so we can manipulate it without changing the original data.

# Now that we have the dataframe, lets go through each row and column, and check for missing values.
# Notabley, here face our first question. Do we just delete invalid rows, or give them a default value?
# For now, I think we will give invalid/empty values a default of -1 for numeric, or N/A for string values. 

df = df.replace(r"^\s*$", np.nan, regex=True) # Replace any empty values with NaN.

cols_str = df.select_dtypes(include=["object", "string"]).columns # Get all the string columns.
cols_num = df.select_dtypes(include=["number"]).columns # Get all the numeric columns.

df[cols_str] = df[cols_str].fillna("N/A") # Fill all string columns with "N/A" for NaN values.

df.isna().sum().sum() # checks that all those empty values are filled


In [ ]:
# Now that we have imported the dataset and filled in the missing values, lets do a few checks:

# 1. Set the numeric columns to numeric (in case some were saved as string but still are "123", etc)
numeric_features = ["id", "mass (g)", "reclat", "reclong"]

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce") # convers to numeric, NaN if invalid. will set to -1 again at the end.

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")# converts year to integers.

# 2. Check again for NaN values after our conversions, and fill with default vals if needed.
changed_cols = ["id", "mass (g)", "reclat", "reclong"]
print(df[changed_cols].isna().sum()) # checked for NaN values, was 0! so we can continue.

# 3. Ensuring our coordinates and mass columns are within the correct ranges:
invalid_lat = df["reclat"].notna() & ~df["reclat"].between(-90, 90)
invalid_lon = df["reclong"].notna() & ~df["reclong"].between(-180, 180)

print("Invalid latitude rows:", invalid_lat.sum())
print("Invalid longitude rows:", invalid_lon.sum())

bad_idx = df.index[invalid_lon]
df.loc[bad_idx, ["id", "name", "reclat", "reclong", "GeoLocation"]]

# Interesting! Notice we have a single invalid longitude value across the entire dataset. Its 354.47333, 
# for the meteor "Meridiani Planum", a meteor that was discovered on mars. Was this a naming mistake? Interesting.
# I wont edit this row for now, since it is worth noting. 

In [ ]:
# Next lets find and get rid of any duplicate rows, if any.
duplicates_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicates_count)

df = df.drop_duplicates().reset_index(drop=True) # Drop duplicate rows if any.

print("Dataset size after removing dupliates:", len(df))
print("Remaining duplicates:", df.duplicated().sum()) # Check again for duplicates, should be 0 now.   

# Well this cell was entirely pointless, as there was not a single duplicate. wow!
# However, we should check for duplicate IDs specifically, since they are supposed to be unique.
dup_id_rows = df.duplicated(subset=["id"], keep=False).sum()
print("Rows with duplicate IDs:", dup_id_rows)

df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

print("Dataset size after removing dupliates:", len(df))
print("Remaining duplicate IDs:", df.duplicated(subset=["id"]).sum())

# Nope! no duplicate ID's!

In [ ]:
# Finally, lets save our cleaned dataset to our processed data folder
df.to_csv("../data/processed/meteorite_landings_clean.csv", index=False)